# TAMP-OS — Lithograph Pipeline
**Extends the TAMP workflow to replace Bambu Lab with a fully open-source pipeline.**

```
Microscopy Image → Height Map → STL → G-code (PrusaSlicer) → Printer
```

### Install dependencies (run once)

In [ ]:
!pip install numpy pillow scipy numpy-stl

### Imports

In [ ]:
import argparse
import subprocess
import sys
from pathlib import Path

import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh

---
## Step 1 — Image → Height Map

In [ ]:
def image_to_heightmap(
    image_path,
    output_size=(512, 512),
    invert=False,
    blur_sigma=1.0,
    contrast_percentile=(2.0, 98.0),
    preserve_aspect=True,
    flip=True,
):
    """
    Convert a microscopy image to a normalised [0, 1] height map.

    Args:
        image_path:          Path to input image (any format PIL supports).
        output_size:         (width, height) in pixels for the height map.
                             If preserve_aspect=True, height is derived from
                             the image aspect ratio and output_size[0] is
                             used as the reference width.
        invert:              Flip dark/light mapping (useful for SEM images
                             where bright = high structure).
        blur_sigma:          Gaussian blur sigma to smooth noise before
                             height mapping.
        contrast_percentile: Low/high percentile clipping for contrast
                             stretching.
        preserve_aspect:     If True (default), keeps the original image
                             aspect ratio. Match --size-x / --size-y accordingly.
        flip:                If True (default), flips the height map vertically
                             so the print orientation matches the original image.
                             Important for images with text or directional features.

    Returns:
        2-D float32 numpy array with values in [0, 1].
    """
    img = Image.open(image_path).convert("L")
    orig_w, orig_h = img.size

    if preserve_aspect and (orig_w != orig_h):
        target_w = output_size[0]
        target_h = round(target_w * orig_h / orig_w)
        actual_size = (target_w, target_h)
        print(
            f"[i] Image is {orig_w}x{orig_h} (non-square, ratio "
            f"{orig_w/orig_h:.3f}). Height map will be {target_w}x{target_h}.\n"
            f"    Make sure size_x and size_y preserve this ratio,\n"
            f"    e.g. size_x=100, size_y={round(100*orig_h/orig_w, 1)}\n"
            f"    otherwise the lithograph will appear stretched."
        )
    else:
        actual_size = output_size

    img = img.resize(actual_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)

    lo = np.percentile(arr, contrast_percentile[0])
    hi = np.percentile(arr, contrast_percentile[1])
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)

    if blur_sigma > 0:
        arr = gaussian_filter(arr, sigma=blur_sigma)

    if invert:
        arr = 1.0 - arr

    if flip:
        arr = np.flipud(arr)

    return arr

---
## Step 2 — Height Map → STL

In [ ]:
def heightmap_to_stl(
    heightmap,
    output_path,
    base_thickness_mm=1.0,
    max_relief_mm=3.0,
    physical_size_mm=(100.0, 100.0),
):
    """
    Convert a [0,1] height map to a solid STL mesh suitable for FDM printing.

    The mesh has:
      - A top surface shaped by the height map.
      - A flat bottom at z = 0.
      - Four side walls.

    Args:
        heightmap:           2-D float32 array, values in [0, 1].
        output_path:         Where to save the .stl file.
        base_thickness_mm:   Minimum solid base below the relief.
        max_relief_mm:       Height difference between lowest and highest point.
        physical_size_mm:    (x_mm, y_mm) real-world footprint of the print.

    Returns:
        Path to the written STL file.
    """
    rows, cols = heightmap.shape
    dx = physical_size_mm[0] / (cols - 1)
    dy = physical_size_mm[1] / (rows - 1)

    # Aspect ratio check
    hm_ratio = cols / rows
    print_ratio = physical_size_mm[0] / physical_size_mm[1]
    if abs(hm_ratio - print_ratio) > 0.05:
        print(
            f"[WARNING] Aspect ratio mismatch!\n"
            f"    Height map is {cols}x{rows} (ratio {hm_ratio:.3f})\n"
            f"    Print size is {physical_size_mm[0]}x{physical_size_mm[1]} mm "
            f"(ratio {print_ratio:.3f})\n"
            f"    The lithograph will appear stretched. "
            f"Consider size_y={round(physical_size_mm[0] / hm_ratio, 1)}"
        )

    z_top = base_thickness_mm + heightmap * max_relief_mm
    num_top_tris = (rows - 1) * (cols - 1) * 2
    num_bottom_tris = (rows - 1) * (cols - 1) * 2
    num_side_tris = 2 * ((rows - 1) + (cols - 1)) * 2
    total_tris = num_top_tris + num_bottom_tris + num_side_tris
    litho_mesh = mesh.Mesh(np.zeros(total_tris, dtype=mesh.Mesh.dtype))
    tri_idx = 0

    def add_tri(v0, v1, v2):
        nonlocal tri_idx
        litho_mesh.vectors[tri_idx] = [v0, v1, v2]
        tri_idx += 1

    for r in range(rows - 1):
        for c in range(cols - 1):
            x0, y0 = c * dx, r * dy
            x1 = (c + 1) * dx
            x2, y2 = c * dx, (r + 1) * dy
            x3, y3 = (c + 1) * dx, (r + 1) * dy
            z00, z10, z01, z11 = z_top[r,c], z_top[r,c+1], z_top[r+1,c], z_top[r+1,c+1]
            add_tri([x0,y0,z00],[x1,y0,z10],[x2,y2,z01])
            add_tri([x1,y0,z10],[x3,y3,z11],[x2,y2,z01])
            add_tri([x0,y0,0],[x2,y2,0],[x1,y0,0])
            add_tri([x1,y0,0],[x2,y2,0],[x3,y3,0])

    xmax, ymax = (cols-1)*dx, (rows-1)*dy
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx; z0,z1 = z_top[0,c],z_top[0,c+1]
        add_tri([x0,0,0],[x1,0,0],[x0,0,z0]); add_tri([x1,0,0],[x1,0,z1],[x0,0,z0])
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx; z0,z1 = z_top[-1,c],z_top[-1,c+1]
        add_tri([x0,ymax,0],[x0,ymax,z0],[x1,ymax,0]); add_tri([x1,ymax,0],[x0,ymax,z0],[x1,ymax,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy; z0,z1 = z_top[r,0],z_top[r+1,0]
        add_tri([0,y0,0],[0,y0,z0],[0,y1,0]); add_tri([0,y1,0],[0,y0,z0],[0,y1,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy; z0,z1 = z_top[r,-1],z_top[r+1,-1]
        add_tri([xmax,y0,0],[xmax,y1,0],[xmax,y0,z0]); add_tri([xmax,y1,0],[xmax,y1,z1],[xmax,y0,z0])

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    litho_mesh.save(str(output_path))
    print(f"[OK] STL saved -> {output_path}")
    return output_path

---
## Step 3 — STL → G-code via PrusaSlicer CLI

In [ ]:
PRUSASLICER_DEFAULTS = {
    "layer_height": "0.12",
    "first_layer_height": "0.2",
    "fill_density": "15%",
    "fill_pattern": "gyroid",
    "perimeters": "3",
    "support_material": "0",
    "nozzle_diameter": "0.4",
    "filament_type": "PLA",
}

def slice_stl(stl_path, output_dir, prusaslicer_bin="prusa-slicer", printer_profile=None, extra_args=None):
    """
    Slice an STL with PrusaSlicer CLI and return the path to the G-code file.

    prusaslicer_bin paths:
        Linux:   'prusa-slicer'
        macOS:   '/Applications/PrusaSlicer.app/Contents/MacOS/PrusaSlicer'
        Windows: 'C:/Program Files/Prusa3D/PrusaSlicer/prusa-slicer-console.exe'
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    gcode_path = output_dir / (Path(stl_path).stem + ".gcode")
    params = {**PRUSASLICER_DEFAULTS, **(extra_args or {})}
    cmd = [prusaslicer_bin, "--export-gcode", str(stl_path), "--output", str(gcode_path)]
    if printer_profile:
        cmd += ["--load", printer_profile]
    for k, v in params.items():
        cmd += [f"--{k}", str(v)]
    print(f"[->] Slicing: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Slicing failed:\n{result.stderr}")
    print(f"[OK] G-code saved -> {gcode_path}")
    return gcode_path

---
## Step 4 — Send G-code to Printer (Klipper/Moonraker)

In [ ]:
def send_to_klipper(gcode_path, moonraker_host="http://localhost:7125", start_print=False):
    """Upload G-code to a Klipper printer via Moonraker and optionally start the print."""
    try:
        import requests
    except ImportError:
        print("[ERROR] Run: pip install requests")
        return
    filename = Path(gcode_path).name
    with open(gcode_path, "rb") as f:
        response = requests.post(f"{moonraker_host}/server/files/upload", files={"file": (filename, f)})
    if response.status_code != 201:
        raise RuntimeError(f"Upload failed: {response.text}")
    print(f"[OK] Uploaded {filename} to {moonraker_host}")
    if start_print:
        r = requests.post(f"{moonraker_host}/printer/print/start", json={"filename": filename})
        print(f"[OK] Print started" if r.status_code == 200 else f"[ERROR] {r.text}")

---
## ▶ Run the Pipeline

Edit the parameters below and run this cell.

In [ ]:
# ── Parameters ────────────────────────────────────────────
IMAGE_PATH       = "your_image.png"   # path to your SEM/TEM image
OUTPUT_DIR       = "./output"

SIZE_X           = 100.0   # print width in mm
SIZE_Y           = 75.0    # print height in mm (match image aspect ratio!)
BASE_THICKNESS   = 1.0     # solid base below relief (mm)
RELIEF_HEIGHT    = 3.0     # max tactile height difference (mm)
BLUR             = 1.2     # gaussian smoothing (increase for noisy images)
RESOLUTION       = 512     # height map resolution in px
INVERT           = False   # True = dark areas become raised
FLIP             = True    # True = correct orientation (recommended)

# Optional slicing
PRUSASLICER_BIN  = "prusa-slicer"   # path to PrusaSlicer executable
SKIP_SLICE       = True             # set False to auto-slice
LAYER_HEIGHT     = 0.12

# Optional direct printer upload (Klipper only)
MOONRAKER_HOST   = None    # e.g. "http://192.168.1.42:7125"
START_PRINT      = False

# ── Run ───────────────────────────────────────────────────
from pathlib import Path
out_dir = Path(OUTPUT_DIR)

print("\n" + "="*50)
print("  TAMP-OS Lithograph Pipeline")
print("="*50 + "\n")

print("[1/4] Converting image to height map...")
hm = image_to_heightmap(
    IMAGE_PATH, output_size=(RESOLUTION, RESOLUTION),
    invert=INVERT, blur_sigma=BLUR, flip=FLIP,
)
print(f"      Shape: {hm.shape}, range [{hm.min():.3f}, {hm.max():.3f}]")

print("\n[2/4] Generating STL...")
stl_path = out_dir / (Path(IMAGE_PATH).stem + "_lithograph.stl")
heightmap_to_stl(hm, stl_path, base_thickness_mm=BASE_THICKNESS,
                 max_relief_mm=RELIEF_HEIGHT, physical_size_mm=(SIZE_X, SIZE_Y))

if not SKIP_SLICE:
    print("\n[3/4] Slicing...")
    gcode_path = slice_stl(stl_path, out_dir, prusaslicer_bin=PRUSASLICER_BIN,
                           extra_args={"layer_height": str(LAYER_HEIGHT)})
else:
    print("\n[3/4] Skipping slicing.")
    gcode_path = None

if gcode_path and MOONRAKER_HOST:
    print(f"\n[4/4] Uploading to {MOONRAKER_HOST}...")
    send_to_klipper(gcode_path, MOONRAKER_HOST, start_print=START_PRINT)
else:
    print("\n[4/4] Skipping upload.")

print("\n" + "="*50)
print(f"  Done! Files in: {out_dir.resolve()}")
print("="*50)

## Preview — Height Map Visualization

In [ ]:
import matplotlib.pyplot as plt

BASE_MM   = BASE_THICKNESS
RELIEF_MM = RELIEF_HEIGHT

# Use flip=False for display so orientation matches the original image
hm_display = image_to_heightmap(IMAGE_PATH, output_size=(RESOLUTION, RESOLUTION),
                                 blur_sigma=BLUR, flip=False)
stl_height = BASE_MM + hm_display * RELIEF_MM

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#0f0f0f')

orig = np.array(Image.open(IMAGE_PATH).convert('L'))
axes[0].imshow(orig, cmap='gray', aspect='auto')
axes[0].set_title('SEM Image', color='white', fontsize=13, pad=10)
axes[0].axis('off')

im2 = axes[1].imshow(hm_display, cmap='inferno', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Height Map → Normalized [0–1]', color='white', fontsize=13, pad=10)
axes[1].axis('off')
cb2 = plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
cb2.set_label('Normalised height', color='white', fontsize=10)
cb2.ax.yaxis.set_tick_params(color='white', labelcolor='white')

im3 = axes[2].imshow(stl_height, cmap='viridis', aspect='auto', vmin=BASE_MM, vmax=BASE_MM+RELIEF_MM)
axes[2].set_title(f'Lithograph Relief [mm]\n(base {BASE_MM} mm + up to {RELIEF_MM} mm)', color='white', fontsize=13, pad=10)
axes[2].axis('off')
cb3 = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
cb3.set_label('STL height (mm)', color='white', fontsize=10)
cb3.ax.yaxis.set_tick_params(color='white', labelcolor='white')
ticks = [BASE_MM + RELIEF_MM*t for t in [0, 0.25, 0.5, 0.75, 1.0]]
cb3.set_ticks(ticks)
cb3.set_ticklabels([f'{t:.2f} mm' for t in ticks])

plt.tight_layout(pad=2)
plt.show()